<a href="https://colab.research.google.com/github/MelisaYasak/transformer_arch/blob/main/multi_head_attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Multi-head attention, bir cümledeki kelimeler arasındaki ilişkileri tek bir bakış açısından değil, birden fazla farklı bakış açısından aynı anda inceleyen bir mekanizmadır. Model önce her kelimeyi alır ve onu üç farklı role dönüştürür: sorgulayan (Query), eşleşme sağlayan (Key) ve bilgi taşıyan (Value). Daha sonra bu temsilleri parçalara bölerek birden fazla “head” oluşturur; her head, kelimeler arasındaki farklı bir ilişki türünü (örneğin anlam, gramer ya da uzak-bağlantı ilişkileri) öğrenir. Her head kendi içinde attention hesaplamasını yaparak kelimelerin birbirinden ne kadar bilgi alacağını belirler. Sonrasında bu farklı bakış açılarından gelen sonuçlar tekrar birleştirilir ve tek bir vektöre dönüştürülür. Böylece model, bir kelimenin anlamını oluştururken aynı anda birden fazla ilişkiyi dikkate alarak daha zengin ve doğru bir temsil elde eder.

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [8]:

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.size()

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # head'lere böl
        Q = Q.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

        # attention hesapla
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        weights = F.softmax(scores, dim=-1)
        out = torch.matmul(weights, V)

        # tekrar birleştir
        out = out.transpose(1, 2).contiguous().view(B, T, D)

        return self.W_o(out), weights

Bu sınıfa verdiğimiz girdi (x), aslında cümledeki her kelimenin sayısal temsilidir; yani modelin anlayabileceği forma çevrilmiş halidir. Daha açık söylemek gerekirse, bir cümledeki her kelime önce bir embedding vektörüne dönüştürülür ve bu vektörler yan yana gelerek (batch_size, seq_len, d_model) şeklinde bir tensör oluşturur. Burada

* seq_len cümledeki kelime sayısını,

* d_model ise her kelimenin kaç boyutla temsil edildiğini ifade eder.

Multi-head attention bu girdiyi alır çünkü kelimeler arasındaki ilişkileri anlayabilmek için önce onların sayısal temsillerine ihtiyaç duyar; model doğrudan kelimelerle değil, bu vektörlerle çalışır. Bu vektörler sayesinde model, kelimeler arasındaki benzerlikleri ölçebilir, hangi kelimenin hangisiyle daha ilgili olduğunu hesaplayabilir ve sonunda her kelimenin bağlama göre zenginleştirilmiş yeni bir temsilini üretebilir.

In [9]:
torch.manual_seed(42)

d_model = 8
num_heads = 2
seq_len = 3
batch_size = 1

mha = MultiHeadAttention(d_model, num_heads)

# fake input (3 kelime, 8 boyut)
x = torch.rand(batch_size, seq_len, d_model)

output, weights = mha(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Weights shape:", weights.shape)

print("\nOutput:")
print(output)

print("\nAttention Weights:")
print(weights)

Input shape: torch.Size([1, 3, 8])
Output shape: torch.Size([1, 3, 8])
Weights shape: torch.Size([1, 2, 3, 3])

Output:
tensor([[[-0.0049, -0.1396,  0.0071, -0.1801, -0.1216, -0.0497, -0.2109,
           0.5923],
         [-0.0062, -0.1408,  0.0080, -0.1789, -0.1201, -0.0474, -0.2101,
           0.5910],
         [-0.0054, -0.1408,  0.0069, -0.1783, -0.1206, -0.0485, -0.2112,
           0.5906]]], grad_fn=<ViewBackward0>)

Attention Weights:
tensor([[[[0.3110, 0.3459, 0.3432],
          [0.3119, 0.3376, 0.3505],
          [0.3116, 0.3341, 0.3542]],

         [[0.3636, 0.3320, 0.3044],
          [0.3430, 0.3338, 0.3232],
          [0.3495, 0.3361, 0.3144]]]], grad_fn=<SoftmaxBackward0>)


Şimdi outputu değerlendirelim:

Shape’ler:

* Input → (1, 3, 8):
1 cümle, 3 kelime, her kelime 8 boyut
* Output → (1, 3, 8): Her kelimenin güncellenmiş hali
* Weights → (1, 2, 3, 3): 2 head var, her biri 3x3 attention matrisi

Output çıktısında her kelime için “kelimenin bağlama göre güncellenmiş hali”nin vektörünü görmekeyiz. Örneğin bir cümledeki bir kelime, diğer kelimelerden farklı oranlarda bilgi alarak anlamını netleştirir; bu yüzden output vektörleri, modelin cümleyi daha iyi anlamasını sağlayan context-aware (bağlam duyarlı) temsillerdir.

Attention weights ise bu bilginin nasıl paylaştırıldığını gösteren oranlardır. Her weight değeri, bir kelimenin başka bir kelimeden ne kadar bilgi alacağını belirler ve tüm satırın toplamı 1 olacak şekilde normalize edilir. Bu sayede model, hangi kelimenin daha önemli olduğuna karar verir ve bilgiyi buna göre dağıtır. Örneğin bir weight değeri yüksekse, bu o kelimenin diğerine daha fazla katkı sağladığı anlamına gelir. Kısacası weights, “kim ne kadar etkili” sorusunun cevabıdır; output ise bu etkilerin sonucunda ortaya çıkan yeni temsildir.

In [10]:
print(weights[0, 0])
print(weights[0, 1])

tensor([[0.3110, 0.3459, 0.3432],
        [0.3119, 0.3376, 0.3505],
        [0.3116, 0.3341, 0.3542]], grad_fn=<SelectBackward0>)
tensor([[0.3636, 0.3320, 0.3044],
        [0.3430, 0.3338, 0.3232],
        [0.3495, 0.3361, 0.3144]], grad_fn=<SelectBackward0>)


Bu çıktılar, aynı cümle için iki farklı attention head’in kelimeler arasındaki ilişkileri nasıl farklı şekillerde değerlendirdiğini gösterir. Her matris 3x3 boyutunda olduğu için 3 kelimelik bir cümleyi temsil eder;
* satırlar “hangi kelime bilgi topluyor” (query),
* sütunlar ise “hangi kelimeden bilgi alıyor” (key) anlamına gelir.

Örneğin ilk head’de (weights[0,0]) birinci satır [0.3110, 0.3459, 0.3432] olduğu için,

* ilk kelime kendisinden %31,
* ikinci kelimeden %34 ve
* üçüncü kelimeden %34 oranında bilgi alarak yeni temsilini oluşturur;

burada ikinci ve üçüncü kelimelerin katkısı biraz daha baskındır. Buna karşılık ikinci head’de (weights[0,1]) ilk satır [0.3636, 0.3320, 0.3044] olduğu için aynı kelime bu kez en çok kendisine (%36) odaklanmakta, diğer kelimelerden daha az bilgi almaktadır.

Benzer şekilde diğer satırlar da her kelimenin bilgi dağılımını gösterir ve dikkat edersen her satırın toplamı 1’dir; bu da bunların birer olasılık dağılımı olduğunu gösterir. İki head arasındaki fark ise kritik noktadır: biri bilgiyi daha dengeli dağıtırken, diğeri belirli kelimelere daha fazla ağırlık verir. Bu sayede model, aynı cümleyi farklı açılardan analiz ederek daha zengin bir bağlam temsili üretir.